In [ ]:
### Import Basic Libraries ###
import numpy as np 
import pandas as pd
from matplotlib import pyplot as plt

In [ ]:
### Taking Dataset ###
df = pd.read_csv('~/Projects/NNFS/train.csv')       # matrix 42,000 x 784
data = np.array(df)
m, n = data.shape       # m = rows, n = column
np.random.seed(4)       # seed so the randomness stays the same every time the code runs

### Data Preprocessing
data_test = data[0:1000].T          # take the first 1000 rows as the test set (matrix (1000 x 784)^T = (784 x 1000))
Y_test = data_test[0]               # separate the label from the features
X_test = data_test[1:n] / 255.0     # separate the features from the label and normalize them to the range 0 - 1

data_train = data[1000:m].T         # use the remaining data as the train set (matrix (41,000 x 784)^T = (784 x 41,000))
Y_train = data_train[0]             # separate the label from the features
X_train = data_train[1:n] / 255.0   # separate the features from the label and normalize them to the range 0 - 1

In [ ]:
### Neural Network ###
class NeuralNetwork:
    ### Parameter Initialization
    def __init__(self, input_size, hidden1_size, hidden2_size, output_size):
        self.W1 = np.random.rand(hidden1_size, input_size) - 0.5                    # hidden layer 1, matrix weight (16, 784)
        self.b1 = np.random.rand(hidden1_size, 1) - 0.5                             # bias matrix (16, 1)
        self.W2 = np.random.rand(hidden2_size, hidden1_size) - 0.5                  # hidden layer 2, matrix weight (16, 16)
        self.b2 = np.random.rand(hidden2_size, 1) - 0.5                             # bias matrix (16, 1)
        self.W3 = np.random.rand(output_size, hidden2_size) - 0.5                   # output layer, matrix weight (10, 16)
        self.b3 = np.random.rand(output_size, 1) - 0.5                              # bias matrix (10, 1)

    ### Activation Function
    def _ReLU(self, Z):
        return np.maximum(0, Z)     # turn negative input into 0, keep positive values unchanged
    
    def _softmax(self, Z):
        return np.exp(Z) / np.sum(np.exp(Z), axis=0)    # exponentiate the input and divide by the total, so all outputs sum to 1

    ### Forward Propagation
    def forwardProp(self, X):
        self.Z1 = self.W1.dot(X) + self.b1
        self.A1 = self._ReLU(self.Z1)
        self.Z2 = self.W2.dot(self.A1) + self.b2
        self.A2 = self._ReLU(self.Z2)
        self.Z3 = self.W3.dot(self.A2) + self.b3
        self.A3 = self._softmax(self.Z3)
        return self.A3
    
    ### Optimization
    def _oneHot(self, Y):                               # One-Hot Encoding
        oneHot_Y = np.zeros((Y.size, Y.max() + 1))            # create a zero matrix with shape (number of samples, max label value + 1)
        oneHot_Y[np.arange(Y.size), Y] = 1                    # set 1 at each (row, column) coordinate, i.e. (index, corresponding label value)
        return oneHot_Y.T
    
    def _derivReLU(self, Z):                            # Derivative of the ReLU function
        return Z > 0
    
    ### Backpropagation (Chain Rule)
    def backProp(self, X, Y):
        # divide by m to get the average gradient instead of the total gradient
        # this keeps weight updates stable even with a large amount of data (using the average instead of total data values)
        m = Y.size                          # Y.size = 41,000
        oneHot_Y = self._oneHot(Y)

        # Layer 3 (Output) - difference between the prediction and the true label
        dZ3 = self.A3 - oneHot_Y            # compute the loss function
        dW3 = 1 / m * dZ3.dot(self.A2.T)
        db3 = 1 / m * np.sum(dZ3, axis=1, keepdims=True)

        # Layer 2 (Hidden) - propagate the error from layer 3, multiplied by the ReLU derivative
        dZ2 = self.W3.T.dot(dZ3) * self._derivReLU(self.Z2)
        dW2 = 1 / m * dZ2.dot(self.A1.T)
        db2 = 1 / m * np.sum(dZ2, axis=1, keepdims=True)

        # Layer 1 (Hidden) - propagate the error from layer 2 to the input
        dZ1 = self.W2.T.dot(dZ2) * self._derivReLU(self.Z1)
        dW1 = 1 / m * dZ1.dot(X.T)
        db1 = 1 / m * np.sum(dZ1, axis=1, keepdims=True)
        return dW1, db1, dW2, db2, dW3, db3
    
    ### Gradient Descent
    def updateParams(self, dW1, db1, dW2, db2, dW3, db3, alpha):
        # subtract the old weights by (learning_rate * gradient)
        # the goal is to move towards the lowest point of the loss function (minimum)
        self.W1 = self.W1 - alpha * dW1
        self.b1 = self.b1 - alpha * db1
        self.W2 = self.W2 - alpha * dW2
        self.b2 = self.b2 - alpha * db2
        self.W3 = self.W3 - alpha * dW3
        self.b3 = self.b3 - alpha * db3

    ### Training Loop
    def train(self, X, Y, epoch, alpha):
        for i in range(epoch):
            self.forwardProp(X)                                         # Step 1: predict (Forward)
            dW1, db1, dW2, db2, dW3, db3 = self.backProp(X, Y)          # Step 2: compute the error (Backprop)
            self.updateParams(dW1, db1, dW2, db2, dW3, db3, alpha)      # Step 3: correct itself (Update)

            if i % 50 == 0:
                predictions = np.argmax(self.A3, axis=0)
                accuracy = np.sum(predictions == Y) / Y.size            # compare the forward propagation result with the label
                print(f"Epoch: {i} to {i+50}")
                print(f"Accuracy: {accuracy:.4f}")

    ### Computing Predictions and Accuracy
    def predict(self, X):
        A3 = self.forwardProp(X)        # A3 contains the probability for each digit
        return np.argmax(A3, axis=0)    # argmax takes the index of the largest value
    
    def getAccuracy(self, X, Y):
        predictions = self.predict(X)
        return np.sum(predictions == Y) / Y.size    # divide by the total number of samples (Y.size) to get an average value (0.0 - 1.0)

In [ ]:
### Defining the Architecture, Training Epochs, and Learning Rate of the Neural Network ###
mlp = NeuralNetwork(input_size=784, hidden1_size=16, hidden2_size=16, output_size=10)
mlp.train(X_train, Y_train, epoch=600, alpha=0.1)

In [ ]:
### Visualizing the Prediction Result ###
def testPrediction(index, model, X, Y):
    img = X[:, index, None]
    prediction = model.predict(img)
    print(f"\nPrediction: {prediction[0]}, Actual Label: {Y[index]}")
    
    # display the image by rescaling the pixel values (0-1 back to 0-255)
    plt.gray()
    plt.imshow(img.reshape(28, 28) * 255, interpolation="nearest")
    plt.show()
    
testPrediction(166, mlp, X_test, Y_test)

In [ ]:
## Comparison between training accuracy and test accuracy
train_acc = mlp.getAccuracy(X_train, Y_train)
test_acc = mlp.getAccuracy(X_test, Y_test)

print(("=" * 5),"Final Result",("=" * 5))
print(f"Akurasi Training: {train_acc:.4f}")
print(f"Akurasi Test: {test_acc:.4f}")
print("=" * 24)